# BÀI TOÁN DỰ BÁO CHUỖI THỜI GIAN VÀ TRIỂN KHAI VỚI MẠNG NƠ-RON HỒI QUY (RNN, LSTM)

Nội dung sổ tay gồm 4 phần trọng tâm:
1. **Các khái niệm cơ bản về RNN, biểu diễn hàm toán học và cài đặt RNN từ đầu bằng NumPy (BPTT)**.
2. **Hai tập dữ liệu chuỗi thời gian: Chứng khoán (Amazon Stock Price) và Giá vàng theo thời gian (Gold Price)**.
3. **Code RNN - PyTorch để dự đoán và triển khai (Deploy Inference Engine / FastAPI)**.
4. **Code RNN - Keras để dự đoán và triển khai (Deploy Inference Engine / FastAPI)**.

## PHẦN 1: CÁC KHÁI NIỆM CƠ BẢN VỀ RNN, BIỂU DIỄN HÀM VÀ CODE SCRATCH

- **Trạng thái ẩn ($h_t$):** Đóng vai trò bộ nhớ nội tại, biểu diễn trạng thái đệ quy qua thời gian.
- **Chia sẻ tham số:** Dùng chung ma trận trọng số $W_{hh}, W_{xh}, W_{hy}$ qua mọi bước thời gian.
- **Biểu diễn hàm toán học:**
  $$a_t = W_{hh} h_{t-1} + W_{xh} x_t + b_h$$
  $$h_t = \tanh(a_t)$$
  $$\hat{y}_t = W_{hy} h_t + b_y$$
- **Lan truyền ngược qua thời gian (BPTT):**
  $$\frac{\partial \mathcal{L}}{\partial W_{hh}} = \sum_{t=1}^T \sum_{k=1}^t \frac{\partial \mathcal{L}_t}{\partial h_t} \frac{\partial h_t}{\partial h_k} \frac{\partial^+ h_k}{\partial W_{hh}}$$
  Hiện tượng triệt tiêu / bùng nổ đạo hàm xuất hiện do tích chuỗi Jacobi $\frac{\partial h_t}{\partial h_k} = \prod_{j=k+1}^t W_{hh}^T \text{diag}(1 - h_j^2)$.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from src.rnn_scratch import VanillaRNNScratch, verify_rnn_scratch

# Kiểm định mô hình RNN tự xây dựng từ đầu (NumPy)
verify_rnn_scratch()

## PHẦN 2: CHỌN 2 TẬP DỮ LIỆU CHUỖI THỜI GIAN, MÔ TẢ VÀ PHÂN BỔ DỮ LIỆU

1. **Tập 1 - Chứng khoán (Amazon Stock Price):** 6,684 phiên giao dịch (1997 - 2023) với các biến: `Open`, `High`, `Low`, `Close`, `Volume`.
2. **Tập 2 - Giá vàng (Gold Price theo thời gian):** 13,320 ngày giao dịch (1968 - 2021) với giá vàng quốc tế.

In [ ]:
import pandas as pd
from src.data_preprocessing import TimeSeriesDataPipeline

# 1. Nạp và mô tả dữ liệu Chứng khoán Amazon
stock_pipe = TimeSeriesDataPipeline(dataset_name='stock', window_size=30)
stock_pipe.load_data()
print("=== THỐNG KÊ MÔ TẢ AMAZON STOCK ===")
print(stock_pipe.get_descriptive_stats()[['mean', 'std', 'min', '50%', 'max', 'skew']])
(X_tr_s, y_tr_s), (X_va_s, y_va_s), (X_te_s, y_te_s), info_s = stock_pipe.temporal_split()
print("Phân bổ dữ liệu Stock (Train/Val/Test):", info_s)

# 2. Nạp và mô tả dữ liệu Giá vàng (Gold Price)
gold_pipe = TimeSeriesDataPipeline(dataset_name='gold', window_size=30)
gold_pipe.load_data()
print("\n=== THỐNG KÊ MÔ TẢ GOLD PRICE ===")
print(gold_pipe.get_descriptive_stats()[['mean', 'std', 'min', '50%', 'max', 'skew']])
(X_tr_g, y_tr_g), (X_va_g, y_va_g), (X_te_g, y_te_g), info_g = gold_pipe.temporal_split()
print("Phân bổ dữ liệu Gold (Train/Val/Test):", info_g)

## PHẦN 3: CODE RNN - PYTORCH ĐỂ DỰ ĐOÁN & DEPLOY

- Kiến trúc `PyTorchRNNModel` hỗ trợ RNN / LSTM / GRU đa tầng với Dropout và Fully Connected Head.
- Huấn luyện với AdamW, Huber Loss, Gradient Clipping chống bùng nổ đạo hàm.
- Đóng gói `PyTorchInferenceEngine` và REST API với FastAPI (dự báo 1 bước và đa bước tự hồi quy).

In [ ]:
import torch
from torch.utils.data import DataLoader
from src.models_pytorch import PyTorchRNNModel, TimeSeriesDataset, train_pytorch_model, evaluate_pytorch_model
from src.deploy_pytorch import PyTorchInferenceEngine

# Huấn luyện và đánh giá trên Amazon Stock
train_loader_s = DataLoader(TimeSeriesDataset(X_tr_s, y_tr_s), batch_size=32, shuffle=True)
val_loader_s = DataLoader(TimeSeriesDataset(X_va_s, y_va_s), batch_size=32, shuffle=False)
test_loader_s = DataLoader(TimeSeriesDataset(X_te_s, y_te_s), batch_size=32, shuffle=False)

pt_stock = PyTorchRNNModel(input_dim=5, hidden_dim=64, num_layers=2, rnn_type='LSTM')
pt_stock, hist_s = train_pytorch_model(pt_stock, train_loader_s, val_loader_s, epochs=15, lr=0.002, save_path='../models/stock_pytorch_lstm.pt')
metrics_pt_s, preds_pt_s, trues_pt_s = evaluate_pytorch_model(pt_stock, test_loader_s, stock_pipe)
print("Kết quả đánh giá PyTorch trên Amazon Stock:", metrics_pt_s)

# Kiểm thử Engine Deploy PyTorch
eng_pt_s = PyTorchInferenceEngine('../models/stock_pytorch_lstm.pt', {'input_dim': 5, 'hidden_dim': 64, 'num_layers': 2, 'rnn_type': 'LSTM', 'target_idx': 3}, '../models/stock_scaler_params.json')
sample_seq_s = X_te_s[0].tolist()
print("Dự báo 1 bước tiếp theo (Stock):", eng_pt_s.predict_next_step(sample_seq_s))
print("Dự báo 5 ngày tiếp theo (Stock Rollout):", eng_pt_s.forecast_multistep(sample_seq_s, steps=5))

## PHẦN 4: CODE RNN - KERAS ĐỂ DỰ ĐOÁN & DEPLOY

- Kiến trúc Keras Sequential API với Stacked LSTM / SimpleRNN / GRU.
- Callbacks chuyên nghiệp: `EarlyStopping`, `ReduceLROnPlateau`, `ModelCheckpoint`.
- Đóng gói `KerasInferenceEngine` và REST API với FastAPI.

In [ ]:
from src.models_keras import build_keras_rnn_model, compile_and_train_keras, evaluate_keras_model
from src.deploy_keras import KerasInferenceEngine

# Huấn luyện và đánh giá trên Gold Price
kr_gold = build_keras_rnn_model(input_shape=(30, 5), hidden_dim=64, num_layers=2, rnn_type='LSTM')
kr_gold, hist_g = compile_and_train_keras(kr_gold, X_tr_g, y_tr_g, X_va_g, y_va_g, epochs=15, lr=0.002, save_path='../models/gold_keras_lstm.keras')
metrics_kr_g, preds_kr_g, trues_kr_g = evaluate_keras_model(kr_gold, X_te_g, y_te_g, gold_pipe)
print("Kết quả đánh giá Keras trên Gold Price:", metrics_kr_g)

# Kiểm thử Engine Deploy Keras
eng_kr_g = KerasInferenceEngine('../models/gold_keras_lstm.keras', '../models/gold_scaler_params.json', target_idx=0)
sample_seq_g = X_te_g[0].tolist()
print("Dự báo 1 bước tiếp theo (Gold):", eng_kr_g.predict_next_step(sample_seq_g))
print("Dự báo 5 ngày tiếp theo (Gold Rollout):", eng_kr_g.forecast_multistep(sample_seq_g, steps=5))